# TP3 — 04: Entrenamiento del modelo de sentimiento

**Modelo elegido:** `TF-IDF + Logistic Regression` (ver `RECOMENDACION_FINAL.md`).

Motivos: es eficiente para 1.600.000 tweets, funciona bien con texto corto, es
interpretable (los coeficientes son pesos por palabra, notebook 06) y constituye un
baseline fuerte y defendible.

## Estrategia de entrenamiento (dos pasos)

1. **Modelo de desarrollo:** split estratificado 90/10 con mezcla aleatoria ->
   permite medir metricas honestas sobre 160.000 tweets que el modelo no vio.
   El shuffle es **critico**: el archivo viene ordenado por clase.
2. **Modelo final:** se reentrena con el **100% del training (1.600.000 tweets)**,
   cumpliendo el requisito mandatorio de la consigna, y se persiste con joblib.

Las predicciones sobre el split de validacion se guardan en
`data/processed/val_predicciones.parquet` para que la notebook 05 evalue sin reentrenar.

In [1]:
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from utils import (
    SEED,
    TARGET,
    TRAIN_CLEAN_PARQUET,
    DATA_PROCESSED,
    MODELS,
    VECTORIZER_JOBLIB,
    MODELO_JOBLIB,
)

In [2]:
train = pd.read_parquet(TRAIN_CLEAN_PARQUET)
assert len(train) == 1_600_000, "El training NO esta completo"
X_texto = train["text_clean"]
y = (train[TARGET] == 4).astype(int)  # 1 = positivo, 0 = negativo
print(f"Corpus completo: {len(X_texto):,} tweets | positivos: {y.mean():.1%}")

Corpus completo: 1,600,000 tweets | positivos: 50.0%


## Split de validacion (estratificado + shuffle)

In [3]:
idx_train, idx_val = train_test_split(
    train.index,
    test_size=0.1,
    stratify=y,
    shuffle=True,       # critico: el archivo esta ordenado por clase
    random_state=SEED,
)
print(f"Desarrollo: {len(idx_train):,} | Validacion: {len(idx_val):,}")
print(f"Proporcion de positivos - desarrollo: {y[idx_train].mean():.4f} | validacion: {y[idx_val].mean():.4f}")

Desarrollo: 1,440,000 | Validacion: 160,000
Proporcion de positivos - desarrollo: 0.5000 | validacion: 0.5000


## Vectorizacion TF-IDF

- `ngram_range=(1, 2)`: el EDA mostro bigramas con mucha carga de sentimiento
  (`cant wait`, `feel better`, `sorry hear`).
- `min_df=5`: descarta n-gramas que aparecen en menos de 5 tweets; controla el
  tamanio del vocabulario y el ruido de terminos unicos.
- `dtype=float32`: reduce la memoria de la matriz a la mitad.
- **Sin remocion de stopwords**: las negaciones (`not`, `no`) discriminan sentimiento.

In [4]:
def nuevo_vectorizador():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, dtype=np.float32)

t0 = time.perf_counter()
vec_dev = nuevo_vectorizador()
X_dev = vec_dev.fit_transform(X_texto[idx_train])
X_val = vec_dev.transform(X_texto[idx_val])
print(f"Vectorizacion (desarrollo): {time.perf_counter() - t0:.0f} s")
print(f"Vocabulario: {len(vec_dev.vocabulary_):,} n-gramas")
print(f"Matriz de desarrollo: {X_dev.shape[0]:,} x {X_dev.shape[1]:,} | densidad {X_dev.nnz / np.prod(X_dev.shape):.2e}")

Vectorizacion (desarrollo): 37 s
Vocabulario: 348,452 n-gramas
Matriz de desarrollo: 1,440,000 x 348,452 | densidad 5.74e-05


## Modelo de desarrollo y metricas de validacion

In [5]:
t0 = time.perf_counter()
lr_dev = LogisticRegression(max_iter=1000, random_state=SEED)
lr_dev.fit(X_dev, y[idx_train])
print(f"Entrenamiento (desarrollo): {time.perf_counter() - t0:.0f} s | iteraciones: {lr_dev.n_iter_[0]}")

Entrenamiento (desarrollo): 16 s | iteraciones: 60


In [6]:
proba_val = lr_dev.predict_proba(X_val)[:, 1]
pred_val = (proba_val >= 0.5).astype(int)

acc = accuracy_score(y[idx_val], pred_val)
f1 = f1_score(y[idx_val], pred_val)
print(f"Validacion (160.000 tweets no vistos): accuracy = {acc:.4f} | F1 = {f1:.4f}")

# Sanity check: TF-IDF + LR en Sentiment140 deberia rondar 0.78-0.83.
assert 0.70 < acc < 0.95, "Accuracy fuera del rango esperable: revisar pipeline"

Validacion (160.000 tweets no vistos): accuracy = 0.8251 | F1 = 0.8266


### Chequeo de overfitting: el mismo modelo en train y validacion

Se evalua el **mismo modelo de desarrollo** sobre su propio split de entrenamiento
(1.440.000 tweets) y sobre validacion (160.000 tweets no vistos). Si train quedara
muy por encima de validacion, el modelo estaria memorizando en lugar de aprender el
patron. Las metricas se persisten para que la notebook 05 las reporte junto al resto
de la evaluacion.

In [7]:
from sklearn.metrics import precision_score, recall_score

proba_train_dev = lr_dev.predict_proba(X_dev)[:, 1]
pred_train_dev = (proba_train_dev >= 0.5).astype(int)


def fila_metricas(conjunto, y_true, y_pred):
    return {
        "conjunto": conjunto,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }


metricas_train_val = pd.DataFrame([
    fila_metricas("train (1.440.000)", y[idx_train], pred_train_dev),
    fila_metricas("validacion (160.000)", y[idx_val], pred_val),
])
metricas_train_val.to_csv(DATA_PROCESSED / "metricas_train_val.csv", index=False)

brecha = metricas_train_val.loc[0, "accuracy"] - metricas_train_val.loc[1, "accuracy"]
print(f"Brecha de accuracy train - validacion: {brecha:.4f}")
metricas_train_val.round(4)

Brecha de accuracy train - validacion: 0.0231


,conjunto,accuracy,precision,recall,f1
0,train (1.440.000),0.8482,0.8420,0.8573,0.8495
1,validacion (160.000),0.8251,0.8195,0.8338,0.8266


In [8]:
# Se persisten las predicciones de validacion para que la notebook 05 evalue
# en detalle (metricas, matriz de confusion, ejemplos de errores) sin reentrenar.
val_pred = pd.DataFrame({
    "idx": idx_val,
    "y_true": y[idx_val].to_numpy(),
    "proba_pos": proba_val,
})
val_pred.to_parquet(DATA_PROCESSED / "val_predicciones.parquet", index=False)
print(f"Guardado: val_predicciones.parquet ({len(val_pred):,} filas)")

Guardado: val_predicciones.parquet (160,000 filas)


## Modelo final con el 100% de los datos (requisito mandatorio)

El modelo que se persiste y se usa contra el test manual (notebook 05) y para la
interpretacion (notebook 06) se entrena con **todos** los tweets del training.

In [9]:
t0 = time.perf_counter()
vec_final = nuevo_vectorizador()
X_full = vec_final.fit_transform(X_texto)
lr_final = LogisticRegression(max_iter=1000, random_state=SEED)
lr_final.fit(X_full, y)
print(f"Entrenamiento final: {time.perf_counter() - t0:.0f} s")
print(f"El modelo final fue entrenado con {X_full.shape[0]:,} tweets (dataset completo)")
print(f"Vocabulario final: {len(vec_final.vocabulary_):,} n-gramas")
assert X_full.shape[0] == 1_600_000

Entrenamiento final: 59 s
El modelo final fue entrenado con 1,600,000 tweets (dataset completo)
Vocabulario final: 379,089 n-gramas


In [10]:
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(vec_final, VECTORIZER_JOBLIB, compress=3)
joblib.dump(lr_final, MODELO_JOBLIB, compress=3)
for p in (VECTORIZER_JOBLIB, MODELO_JOBLIB):
    print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")

tfidf_vectorizer.joblib: 4.2 MB
logistic_regression.joblib: 1.4 MB


## Resumen

- Split 90/10 estratificado con shuffle (el archivo venia ordenado por clase).
- TF-IDF de unigramas + bigramas (`min_df=5`, float32), sin remover stopwords.
- Logistic Regression como baseline interpretable.
- Accuracy de validacion dentro del rango esperable para este dataset (~0,80).
- **Chequeo de overfitting**: el mismo modelo de desarrollo se evalua en su train y
  en validacion; las metricas quedan persistidas y la notebook 05 las compara.
- **Modelo final entrenado con los 1.600.000 tweets** y persistido en `models/`.
- Evaluacion detallada en la notebook 05; interpretacion en la 06.